# Wahapedia 40k Scraper
Run cells top to bottom. Each cell is self-contained for easy debugging.

In [3]:
# Cell 1 — Install dependencies (only need to run once)
!pip install selenium beautifulsoup4 lxml webdriver-manager -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
httpcore 1.0.7 requires h11<0.15,>=0.13, but you have h11 0.16.0 which is incompatible.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.2.6 which is incompatible.
tensorflow 2.19.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.3, but you have protobuf 3.20.1 which is incompatible.
transformers 4.49.0 requires huggingface-hub<1.0,>=0.26.0, but you have huggingface-hub 1.14.0 which is incompatible.
tokenizers 0.21.0 requires huggingface-hub<1.0,>=0.16.4, but you have huggingface-hub 1.14.0 which is incompatible.
ydata-profiling 4.12.2 requires numpy<2.2,>=1.16.0, but you have numpy 2.2.6 which is incompatible.
chasm 0.1.0 requires numpy<2, but you have numpy 2.2.6 which is incompatible.
tf-keras 2.16.0 requires tensorflow<2.17,>=2.16, but you have tenso

In [4]:
# Cell 2 — Imports
import time, re, json
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

print('Imports OK')

Imports OK


/home/mat2m10/.pyenv/versions/3.12.9/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.7.0) or chardet (7.4.3)/charset_normalizer (3.4.1) doesn't match a supported version!
  warnings.warn(


In [6]:
# Cell — Try plain requests first (no browser needed)
import requests
from bs4 import BeautifulSoup

FACTION_SLUG = 'space-marines'
URL = f'https://wahapedia.ru/wh40k10ed/factions/{FACTION_SLUG}/datasheets.html'

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0 Safari/537.36',
    'Accept-Language': 'en-US,en;q=0.9',
}

r = requests.get(URL, headers=headers, timeout=15)
print(f'Status: {r.status_code}')

soup = BeautifulSoup(r.text, 'lxml')

# Quick check: how much content did we actually get?
all_text = soup.get_text()
print(f'Page text length: {len(all_text)} chars')
print('\nFirst 500 chars:')
print(all_text[:500])

Status: 200
Page text length: 3787538 chars

First 500 chars:




































Datasheets















 





 

AuthorCommunityData ExportSubscribe



 

Settings

Show fluff text



Show Legends and not recommended rules/datasheets



Show Forge World datasheets



Show Crusade Rules



Show Boarding Actions rules



Show base size



Show Core Stratagems



Show datasheet features (Stratagems, Enhancements, etc.)

Color theme: 
	
OS default
Bright
Dark





The Rules

Quick Start GuideCore RulesCrusade RulesRules Commentary


In [5]:
# Cell 3 — Launch browser and fetch one faction page
# Change FACTION_SLUG to whichever faction you want
FACTION_SLUG = 'space-marines'
URL = f'https://wahapedia.ru/wh40k10ed/factions/{FACTION_SLUG}/datasheets.html'

opts = Options()
opts.add_argument('--headless=new')
opts.add_argument('--no-sandbox')
opts.add_argument('--disable-dev-shm-usage')
opts.add_argument('--window-size=1920,1080')
opts.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36')

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=opts
)

driver.get(URL)
print(f'Page title: {driver.title}')
print('Waiting for JS to render...')
time.sleep(4)  # give the page time to fully render
print('Done.')

SessionNotCreatedException: Message: session not created
from unknown error: cannot find Chrome binary; For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#sessionnotcreatedexception
Stacktrace:
#0 0x6282e707814a <unknown>
#1 0x6282e6a5aff9 <unknown>
#2 0x6282e6a96956 <unknown>
#3 0x6282e6a94259 <unknown>
#4 0x6282e6ae3c36 <unknown>
#5 0x6282e6ae331c <unknown>
#6 0x6282e6aa2cb2 <unknown>
#7 0x6282e6aa3ac1 <unknown>
#8 0x6282e703ebd7 <unknown>
#9 0x6282e703d3d9 <unknown>
#10 0x6282e70280e6 <unknown>
#11 0x6282e703df7a <unknown>
#12 0x6282e7010080 <unknown>
#13 0x6282e7064df8 <unknown>
#14 0x6282e7064f95 <unknown>
#15 0x6282e7076cce <unknown>
#16 0x7295a2c9caa4 <unknown>
#17 0x7295a2d29c6c <unknown>


In [ ]:
# Cell 4 — Inspect raw HTML to find the right CSS classes
# This is the key diagnostic step before writing any parsing logic
soup = BeautifulSoup(driver.page_source, 'lxml')

# Print a snippet of the page to see what we're working with
# Look for divs that look like datasheet containers
all_divs = soup.find_all('div', class_=True)

# Collect unique class names that appear in the page
class_counts = {}
for tag in soup.find_all(class_=True):
    for cls in tag.get('class', []):
        class_counts[cls] = class_counts.get(cls, 0) + 1

# Show classes that appear many times (likely repeating components like unit cards)
print('=== Frequently repeating CSS classes (likely unit/weapon rows) ===')
for cls, count in sorted(class_counts.items(), key=lambda x: -x[1])[:40]:
    print(f'  {count:4d}x  .{cls}')

In [ ]:
# Cell 5 — Narrow down: find classes that look like datasheets
# After seeing the output of Cell 4, look for 'datasheet', 'unit', 'card' etc.
# UPDATE these patterns based on what you saw above
DATASHEET_PATTERN = re.compile(r'datasheet|Datasheet|unitCard|unit-card', re.I)

candidates = soup.find_all('div', class_=DATASHEET_PATTERN)
print(f'Found {len(candidates)} potential datasheet containers')

if candidates:
    # Show the first one so we can inspect its structure
    first = candidates[0]
    print('\n=== First container HTML (truncated) ===')
    print(str(first)[:3000])

In [ ]:
# Cell 6 — Extract unit name from the first datasheet
# UPDATE the selector based on what you saw in Cell 5
if candidates:
    ds = candidates[0]

    # Try common heading patterns
    for tag in ['h1', 'h2', 'h3', 'h4']:
        headings = ds.find_all(tag)
        if headings:
            print(f'  {tag} tags found: {[h.get_text(strip=True) for h in headings]}')

In [ ]:
# Cell 7 — Extract stat block (M / T / SV / W / LD / OC)
# Inspect what classes the stat table uses
if candidates:
    ds = candidates[0]

    # Print all child tags with their classes to identify the stat row
    print('=== All tags inside first datasheet container ===')
    for child in ds.descendants:
        if hasattr(child, 'name') and child.name and child.get('class'):
            text = child.get_text(strip=True)[:60]
            print(f'  <{child.name}> class={child["class"]}  text={text!r}')

In [ ]:
# Cell 8 — Once you've identified the classes, write targeted extraction
# !! UPDATE the class names below based on Cells 5-7 inspection !!

def extract_unit(ds):
    """Extract a single datasheet. Adjust class names after inspecting HTML."""
    result = {}

    # --- Name ----------------------------------------------------------------
    # EXAMPLE: name_tag = ds.find('div', class_='dsTitle1')  <- update this
    name_tag = ds.find(re.compile(r'h[1-4]'))  
    result['name'] = name_tag.get_text(strip=True) if name_tag else 'Unknown'

    # --- Stats ---------------------------------------------------------------
    # EXAMPLE: stat rows often look like <td class="char-value">6"</td>
    stats = {}
    stat_labels = ds.find_all(class_=re.compile(r'char.*label|stat.*label|label', re.I))
    stat_values = ds.find_all(class_=re.compile(r'char.*value|stat.*value|value', re.I))
    for label, value in zip(stat_labels, stat_values):
        k = label.get_text(strip=True)
        v = value.get_text(strip=True)
        if k:
            stats[k] = v
    result['stats'] = stats

    # --- Weapons -------------------------------------------------------------
    weapons = []
    # Weapon tables typically have 7-8 columns: name, range, A, BS/WS, S, AP, D, abilities
    for table in ds.find_all('table'):
        rows = table.find_all('tr')
        for row in rows[1:]:  # skip header row
            cells = [td.get_text(strip=True) for td in row.find_all('td')]
            if len(cells) >= 7:
                weapons.append({
                    'name':      cells[0],
                    'range':     cells[1],
                    'A':         cells[2],
                    'BS_WS':     cells[3],
                    'S':         cells[4],
                    'AP':        cells[5],
                    'D':         cells[6],
                    'abilities': cells[7] if len(cells) > 7 else '',
                })
    result['weapons'] = weapons

    # --- Keywords ------------------------------------------------------------
    kw_tag = ds.find(class_=re.compile(r'keyword', re.I))
    if kw_tag:
        raw = kw_tag.get_text(strip=True)
        raw = re.sub(r'^KEYWORDS\s*:?\s*', '', raw, flags=re.I)
        result['keywords'] = [k.strip() for k in raw.split(',') if k.strip()]
    else:
        result['keywords'] = []

    return result


# Test on first unit
if candidates:
    unit = extract_unit(candidates[0])
    print(json.dumps(unit, indent=2))

In [ ]:
# Cell 9 — Scrape all units on the page
all_units = [extract_unit(ds) for ds in candidates]
print(f'Scraped {len(all_units)} units')
for u in all_units:
    print(f"  {u['name']}  |  stats={list(u['stats'].keys())}  |  weapons={len(u['weapons'])}")

In [ ]:
# Cell 10 — Save to JSON
import json
out_path = f'{FACTION_SLUG}_units.json'
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(all_units, f, indent=2, ensure_ascii=False)
print(f'Saved to {out_path}')

In [ ]:
# Cell 11 — Always close the driver when done
driver.quit()
print('Browser closed.')